#### **1. INITIALIZATION**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
from transformers import CLIPVisionModel, CLIPImageProcessor
import shutil
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

#### **2. LOAD MODEL AND NEW ARCHITECTURE**

In [4]:
# 1. Define the modified architecture
class ClassificationCLIP(nn.Module):
    def __init__(self, model_path):
        super(ClassificationCLIP, self).__init__()

        print("Loading CLIP Vision Encoder...")
        self.vision_encoder = CLIPVisionModel.from_pretrained(model_path)
        hidden_size = self.vision_encoder.config.hidden_size

        print("Attaching Classification Head...")
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, pixel_values):
        outputs = self.vision_encoder(pixel_values=pixel_values)
        pooled_output = outputs.pooler_output
        logits = self.classifier(pooled_output)
        return logits

# 2. Paths configuration
model_path = "/content/drive/MyDrive/Model/clip_model"
csv_path = "/content/drive/MyDrive/TrainingData/dataset_inventory.csv"
trained_folder = "/content/drive/MyDrive/Model/clip_classification_weights"
os.makedirs(trained_folder, exist_ok=True)
best_weights_path = os.path.join(trained_folder, "best_classifier_weights.pth")

# 3. Load Model and Processor
print("Initializing Model and Processor...")
my_model = ClassificationCLIP(model_path)
processor = CLIPImageProcessor.from_pretrained(model_path)

# 4. Move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
my_model = my_model.to(device)
print(f"Model loaded and moved to {device}")

Initializing Model and Processor...
Loading CLIP Vision Encoder...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: /content/drive/MyDrive/Model/clip_model
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
visual_projection.weight                                     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.o

Attaching Classification Head...
Model loaded and moved to cuda


#### **3. SAMPLE 5K DATA FROM DATASET (SAVE ON COMPUTER LOCAL C DRIVE, NOT ON GG DRIVE)**

In [6]:
# # ========================================================
# # GIAI ĐOẠN 1: LẤY MẪU VÀ ĐÁNH DẤU VÀO FILE GỐC
# # ========================================================

# # 1. Đọc file CSV chứa toàn bộ 28k ảnh
# csv_path = "/content/drive/MyDrive/TrainingData/dataset_inventory.csv"
# df = pd.read_csv(csv_path)

# # 2. Hàm lấy mẫu 5000 ảnh (4000 Train, 500 Val, 500 Test)
# # Việc bốc ngẫu nhiên (sample) từ các tập đã chia sẽ tự động giữ được sự cân bằng giữa ảnh Real/Fake
# def get_sample(group):
#     if group.name == 'train':
#         return group.sample(n=4000, random_state=42)
#     elif group.name == 'val':
#         return group.sample(n=500, random_state=42)
#     elif group.name == 'test':
#         return group.sample(n=500, random_state=42)
#     return group

# # Lấy ra các index của 5000 ảnh được chọn
# sampled_df = df.groupby('split', group_keys=False).apply(get_sample)
# sampled_indices = sampled_df.index

# # 3. Tạo cột 'sample_data': Đánh dấu 1 cho 5000 ảnh được chọn, 0 cho các ảnh còn lại
# df['sample_data'] = 0
# df.loc[sampled_indices, 'sample_data'] = 1

# # Lưu lại file CSV gốc để ghi nhận cột mới
# df.to_csv(csv_path, index=False)
# print(f"✅ Đã đánh dấu {len(sampled_indices)} ảnh vào cột 'sample_data' và lưu đè lên file: {csv_path}")

In [5]:
# ========================================================
# GIAI ĐOẠN 2: COPY DATA ĐƯỢC ĐÁNH DẤU XUỐNG LOCAL DISK
# ========================================================

local_sample_dir = "/content/Local_5K_Sample"
local_sample_csv = "/content/local_5k_inventory.csv"
os.makedirs(local_sample_dir, exist_ok=True)

csv_path = "/content/drive/MyDrive/TrainingData/dataset_inventory.csv"
df = pd.read_csv(csv_path)

# 4. Lọc ra đúng 5000 ảnh đã được đánh dấu
df_sample_only = df[df['sample_data'] == 1].copy()

# Hàm copy từng file
def copy_single_image(row_data):
    index, row = row_data
    src_path = row['file_path']

    # Tạo tên file an toàn (thêm index vào trước để tránh trùng tên file từ các thư mục khác nhau)
    safe_filename = f"{index}_{row['file_name']}"
    dest_path = os.path.join(local_sample_dir, safe_filename)

    # Copy nếu chưa tồn tại
    if not os.path.exists(dest_path):
        try:
            shutil.copy2(src_path, dest_path)
        except Exception as e:
            return None # Bỏ qua nếu file gốc bị lỗi hoặc không tồn tại

    return dest_path

print(f"🚀 Đang copy {len(df_sample_only)} ảnh sang Local Disk ({local_sample_dir})...")
rows = list(df_sample_only.iterrows())

# 5. Dùng đa luồng (Multi-threading) để copy siêu tốc (thay vì copy từng file một)
with ThreadPoolExecutor(max_workers=16) as executor:
    new_paths = list(tqdm(executor.map(copy_single_image, rows), total=len(df_sample_only)))

# 6. Cập nhật cột file_path trỏ về ổ Local
df_sample_only['file_path'] = new_paths
df_sample_only = df_sample_only.dropna(subset=['file_path']) # Xóa các dòng bị lỗi không copy được

# Lưu thành 1 file CSV mới (Local CSV) chuyên dùng để Train siêu tốc
df_sample_only.to_csv(local_sample_csv, index=False)
print(f"🎉 Hoàn tất! File CSV local chuẩn bị cho DataLoader đã lưu tại: {local_sample_csv}")

🚀 Đang copy 5000 ảnh sang Local Disk (/content/Local_5K_Sample)...


  0%|          | 0/5000 [00:00<?, ?it/s]

🎉 Hoàn tất! File CSV local chuẩn bị cho DataLoader đã lưu tại: /content/local_5k_inventory.csv


#### **4. DATASET & DATALOADER**

In [6]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms # Vũ khí thay thế Hugging Face Processor

# 1. Cấu hình đường dẫn
local_csv_path = "/content/local_5k_inventory.csv"

# 2. Bộ tiền xử lý ảnh C++ siêu tốc (Chuẩn hóa đúng theo thông số của CLIP)
clip_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                         std=[0.26862954, 0.26130258, 0.27577711])
])

# 3. Định nghĩa Dataset mới
class FastDeepfakeDataset(Dataset):
    def __init__(self, csv_file, split_name, transform):
        self.df = pd.read_csv(csv_file)
        self.df = self.df[self.df['split'] == split_name].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'file_path']
        label = float(self.df.loc[idx, 'real_image'])
        subgroup = self.df.loc[idx, 'label']

        try:
            with Image.open(img_path) as img:
                image = img.convert('RGB')
                # Áp dụng bộ xử lý siêu tốc ngay lập tức
                pixel_values = self.transform(image)
        except Exception:
            # Nếu file lỗi, trả về ảnh đen an toàn
            pixel_values = torch.zeros((3, 224, 224))

        label_tensor = torch.tensor([label], dtype=torch.float32)
        return pixel_values, label_tensor, subgroup

# 4. Khởi tạo DataLoaders
batch_size = 32
print("Đang khởi tạo Dataloaders Siêu Tốc từ ổ Local...")

train_dataset = FastDeepfakeDataset(local_csv_path, 'train', clip_transform)
val_dataset = FastDeepfakeDataset(local_csv_path, 'val', clip_transform)
test_dataset = FastDeepfakeDataset(local_csv_path, 'test', clip_transform)

# Bật persistent_workers=True để triệt tiêu lỗi chữ đỏ bám dai dẳng
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)

print(f"✅ Dữ liệu sẵn sàng! Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Đang khởi tạo Dataloaders Siêu Tốc từ ổ Local...
✅ Dữ liệu sẵn sàng! Train: 4000 | Val: 500 | Test: 500


#### **4. TRAINING LOOP**

In [7]:
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm
import torch

# ========================================================
# BƯỚC 1: CẤU HÌNH "RÃ ĐÔNG" NÃO BỘ CLIP
# ========================================================
# 1. Đóng băng toàn bộ mô hình trước để đảm bảo an toàn
for param in my_model.vision_encoder.parameters():
    param.requires_grad = False

# 2. Mở khóa (Unfreeze) 2 lớp Transformer cuối cùng của lõi CLIP
# Điều này ép GPU phải cập nhật tạ và giúp mô hình tự học đặc trưng Deepfake
for layer in my_model.vision_encoder.vision_model.encoder.layers[-2:]:
    for param in layer.parameters():
        param.requires_grad = True

# 3. Mở khóa lớp Phân loại (Classifier)
for param in my_model.classifier.parameters():
    param.requires_grad = True

# In báo cáo sức mạnh
trainable_params = sum(p.numel() for p in my_model.parameters() if p.requires_grad)
print(f"🔥 SỐ LƯỢNG THAM SỐ SẼ ĐƯỢC HUẤN LUYỆN: {trainable_params:,}")

# ========================================================
# BƯỚC 2: CẤU HÌNH HÀM LOSS, OPTIMIZER & AMP
# ========================================================
criterion = nn.BCEWithLogitsLoss()

# Lọc chỉ truyền những tham số được mở khóa vào Optimizer, dùng Learning Rate nhỏ (2e-5) để fine-tune an toàn
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, my_model.parameters()), lr=2e-5, weight_decay=1e-2)

# Khởi tạo siêu vũ khí AMP để tiết kiệm 50% VRAM
scaler = torch.amp.GradScaler('cuda')

num_epochs = 5
best_val_loss = float('inf')

# ========================================================
# BƯỚC 3: VÒNG LẶP HUẤN LUYỆN CHÍNH
# ========================================================
print("\n🚀 BẮT ĐẦU FINE-TUNING (TỐC ĐỘ MAX + GPU HOẠT ĐỘNG)...")
for epoch in range(num_epochs):
    print(f"\n--- Epoch {epoch+1}/{num_epochs} ---")

    # -----------------------
    # GIAI ĐOẠN 1: TRAINING
    # -----------------------
    my_model.train()
    train_loss = 0.0

    train_bar = tqdm(train_loader, desc=f"Training Epoch {epoch+1}")
    for images, labels, _ in train_bar:
        # non_blocking=True giúp luân chuyển data mượt hơn
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # Chạy thuật toán trong môi trường Float16
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            outputs = my_model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        train_bar.set_postfix(loss=loss.item())

        # Xóa rác ngay lập tức
        del images, labels, outputs, loss

    avg_train_loss = train_loss / len(train_loader)

    # -----------------------
    # GIAI ĐOẠN 2: VALIDATION
    # -----------------------
    my_model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels, _ in tqdm(val_loader, desc=f"Validating Epoch {epoch+1}"):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(device_type='cuda', dtype=torch.float16):
                outputs = my_model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item()

            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            del images, labels, outputs, loss

    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = correct / total * 100

    print(f"✅ Tổng kết Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Accuracy: {val_accuracy:.2f}%")

    # Lưu Model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        # Lưu toàn bộ trọng số của mô hình (vì giờ ta đã train cả phần lõi)
        torch.save(my_model.state_dict(), best_weights_path)
        print(f"🏆 [Đã lưu] Model tốt nhất vào: {best_weights_path}")

    # Chỉ dọn cache VRAM nội bộ, KHÔNG dùng gc.collect() để giữ mạng sống cho phụ bếp
    torch.cuda.empty_cache()

print("\n🎉 HOÀN TẤT HUẤN LUYỆN!")

🔥 SỐ LƯỢNG THAM SỐ SẼ ĐƯỢC HUẤN LUYỆN: 25,193,473

🚀 BẮT ĐẦU FINE-TUNING (TỐC ĐỘ MAX + GPU HOẠT ĐỘNG)...

--- Epoch 1/5 ---


Training Epoch 1:   0%|          | 0/125 [00:00<?, ?it/s]

Validating Epoch 1:   0%|          | 0/16 [00:00<?, ?it/s]

✅ Tổng kết Epoch 1: Train Loss: 0.3275 | Val Loss: 0.1176 | Val Accuracy: 96.60%
🏆 [Đã lưu] Model tốt nhất vào: /content/drive/MyDrive/Model/clip_classification_weights/best_classifier_weights.pth

--- Epoch 2/5 ---


Training Epoch 2:   0%|          | 0/125 [00:00<?, ?it/s]

...

Validating Epoch 4:   0%|          | 0/16 [00:00<?, ?it/s]

✅ Tổng kết Epoch 4: Train Loss: 0.0026 | Val Loss: 0.0660 | Val Accuracy: 97.80%
🏆 [Đã lưu] Model tốt nhất vào: /content/drive/MyDrive/Model/clip_classification_weights/best_classifier_weights.pth

--- Epoch 5/5 ---


Training Epoch 5:   0%|          | 0/125 [00:00<?, ?it/s]

Validating Epoch 5:   0%|          | 0/16 [00:00<?, ?it/s]

✅ Tổng kết Epoch 5: Train Loss: 0.0005 | Val Loss: 0.0653 | Val Accuracy: 97.80%
🏆 [Đã lưu] Model tốt nhất vào: /content/drive/MyDrive/Model/clip_classification_weights/best_classifier_weights.pth

🎉 HOÀN TẤT HUẤN LUYỆN!


#### **5. EVALUTATION**

In [10]:
from collections import defaultdict
import torch
from tqdm.auto import tqdm

print("\n🔍 ĐANG ĐÁNH GIÁ MÔ HÌNH TỐT NHẤT TRÊN TẬP TEST...")

# 1. Tải trọng số tốt nhất (SỬA LẠI: Nạp tạ cho toàn bộ mô hình thay vì chỉ classifier)
my_model.load_state_dict(torch.load(best_weights_path))
my_model.eval()

# 2. Khởi tạo biến theo dõi
subgroup_correct = defaultdict(int)
subgroup_total = defaultdict(int)
overall_correct = 0
overall_total = 0

# 3. Chạy qua tập Test (Sử dụng các kỹ thuật tối ưu tốc độ)
with torch.no_grad():
    for images, labels, subgroups in tqdm(test_loader, desc="Testing"):
        # Bắn data siêu tốc vào GPU
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Bật chế độ Float16 để inference cực nhanh
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            outputs = my_model(images)
            probs = torch.sigmoid(outputs)

        preds = (probs > 0.5).float()

        # Ma trận so sánh đúng/sai cho cả batch
        correct_tensor = (preds == labels)

        overall_correct += correct_tensor.sum().item()
        overall_total += labels.size(0)

        # Thống kê chi tiết cho từng thư mục/nhóm trong batch
        for i in range(labels.size(0)):
            group_name = subgroups[i]
            subgroup_total[group_name] += 1
            if correct_tensor[i].item():
                subgroup_correct[group_name] += 1

        # Dọn rác VRAM ngay sau khi đếm xong
        del images, labels, outputs, probs, preds, correct_tensor

# 4. In Báo cáo kết quả
overall_accuracy = (overall_correct / overall_total) * 100
print("\n" + "★"*60)
print(f"🏆 ĐỘ CHÍNH XÁC TỔNG THỂ (OVERALL ACCURACY): {overall_accuracy:.2f}% ({overall_correct}/{overall_total})")
print("★"*60)

print("\n📊 CHI TIẾT THEO TỪNG NHÓM CAMERA/THUẬT TOÁN (SUBGROUP PERFORMANCE):")
print("-" * 60)
# Sắp xếp theo tên nhóm cho dễ nhìn
for group in sorted(subgroup_total.keys()):
    correct = subgroup_correct[group]
    total = subgroup_total[group]
    accuracy = (correct / total) * 100
    print(f"Nhóm: {group:<30} | Chính xác: {accuracy:>6.2f}% ({correct}/{total})")
print("-" * 60)

# Xả toàn bộ cache sau khi test xong
torch.cuda.empty_cache()


🔍 ĐANG ĐÁNH GIÁ MÔ HÌNH TỐT NHẤT TRÊN TẬP TEST...


Testing:   0%|          | 0/16 [00:00<?, ?it/s]


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🏆 ĐỘ CHÍNH XÁC TỔNG THỂ (OVERALL ACCURACY): 97.00% (485/500)
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

📊 CHI TIẾT THEO TỪNG NHÓM CAMERA/THUẬT TOÁN (SUBGROUP PERFORMANCE):
------------------------------------------------------------
Nhóm: ai                             | Chính xác:  96.54% (251/260)
Nhóm: nature                         | Chính xác:  97.50% (234/240)
------------------------------------------------------------


In [13]:
import os
import shutil
import pandas as pd
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

# 1. Đọc file CSV gốc trên Drive
drive_csv_path = "/content/drive/MyDrive/TrainingData/dataset_inventory.csv"
df_full = pd.read_csv(drive_csv_path)

# 2. Lọc ra những ảnh CHƯA TỪNG ĐƯỢC CHẠM TỚI (sample_data == 0)
df_unused = df_full[df_full['sample_data'] == 0].copy()

# ======================================================
# CHIẾN THUẬT LẤY MẪU CÂN BẰNG (BALANCED SAMPLING)
# ======================================================
# Tách riêng 2 phe: Thật (Real) và Giả (Fake)
df_real = df_unused[df_unused['real_image'] == 1.0]
df_fake = df_unused[df_unused['real_image'] == 0.0]

# Mục tiêu là lấy 5000 ảnh (2500 Real, 2500 Fake).
# Hàm min() giúp tránh lỗi nếu kho data không còn đủ 2500 ảnh mỗi loại
n_real = min(5000, len(df_real))
n_fake = min(5000, len(df_fake))

# Bốc ngẫu nhiên cho từng tập
sampled_real = df_real.sample(n=n_real, random_state=42)
sampled_fake = df_fake.sample(n=n_fake, random_state=42)

# Gộp lại và xáo trộn đều (shuffle) một lần nữa
df_test_unseen = pd.concat([sampled_real, sampled_fake]).sample(frac=1, random_state=99).reset_index(drop=True)

print(f"Kho dữ liệu chưa dùng còn: {len(df_unused)} ảnh.")
print(f"Đã bốc ra {len(df_test_unseen)} ảnh Super Test gồm: {n_real} ảnh THẬT và {n_fake} ảnh GIẢ.")

# ======================================================
# 4. Copy xuống Local Disk siêu tốc
# ======================================================
local_super_test_dir = "/content/Local_10K_SuperTest"
local_super_test_csv = "/content/local_10k_super_test.csv"
os.makedirs(local_super_test_dir, exist_ok=True)

def copy_single_image(row_data):
    index, row = row_data
    src_path = row['file_path']
    safe_filename = f"unseen_{index}_{row['file_name']}"
    dest_path = os.path.join(local_super_test_dir, safe_filename)

    if not os.path.exists(dest_path):
        try:
            shutil.copy2(src_path, dest_path)
        except Exception:
            return None
    return dest_path

print(f"\n🚀 Đang kéo {len(df_test_unseen)} ảnh cân bằng xuống máy ảo...")
rows = list(df_test_unseen.iterrows())

with ThreadPoolExecutor(max_workers=16) as executor:
    new_paths = list(tqdm(executor.map(copy_single_image, rows), total=len(df_test_unseen)))

# Cập nhật đường dẫn Local và lưu CSV
df_test_unseen['file_path'] = new_paths
df_test_unseen = df_test_unseen.dropna(subset=['file_path'])
df_test_unseen.to_csv(local_super_test_csv, index=False)

print(f"🎉 Đã chuẩn bị xong {len(df_test_unseen)} ảnh Super Test ĐẠT CHUẨN CÂN BẰNG tại Local!")

Kho dữ liệu chưa dùng còn: 23377 ảnh.
Đã bốc ra 10000 ảnh Super Test gồm: 5000 ảnh THẬT và 5000 ảnh GIẢ.

🚀 Đang kéo 10000 ảnh cân bằng xuống máy ảo...


  0%|          | 0/10000 [00:00<?, ?it/s]

🎉 Đã chuẩn bị xong 10000 ảnh Super Test ĐẠT CHUẨN CÂN BẰNG tại Local!


In [14]:
from collections import defaultdict
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# 1. Khởi tạo DataLoader cho tập Super Test
print("Đang nạp tập Super Test vào bộ nhớ...")
# Ta chỉ cần gán tên là 'train', 'val' hay gì cũng được vì ở code trên chưa chia split,
# nhưng để khớp với Dataset class, ta sẽ bỏ qua bước lọc split cho file CSV này.
class SuperTestDataset(FastDeepfakeDataset):
    def __init__(self, csv_file, transform):
        self.df = pd.read_csv(csv_file)
        self.transform = transform

super_test_dataset = SuperTestDataset(local_super_test_csv, clip_transform)
super_test_loader = DataLoader(super_test_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"Đã nạp xong {len(super_test_dataset)} ảnh vào ống phóng!\n")

# 2. Khởi tạo biến đếm
subgroup_correct = defaultdict(int)
subgroup_total = defaultdict(int)
overall_correct = 0
overall_total = 0

# (Đảm bảo mô hình đã nạp bộ tạ tốt nhất và đang ở chế độ eval)
my_model.eval()

# 3. Chạy qua tập Super Test
print("🔍 ĐANG ĐÁNH GIÁ MÔ HÌNH TRÊN 5000 ẢNH HOÀN TOÀN MỚI...")
with torch.no_grad():
    for images, labels, subgroups in tqdm(super_test_loader, desc="Super Testing"):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type='cuda', dtype=torch.float16):
            outputs = my_model(images)
            probs = torch.sigmoid(outputs)

        preds = (probs > 0.5).float()
        correct_tensor = (preds == labels)

        overall_correct += correct_tensor.sum().item()
        overall_total += labels.size(0)

        for i in range(labels.size(0)):
            group_name = subgroups[i]
            subgroup_total[group_name] += 1
            if correct_tensor[i].item():
                subgroup_correct[group_name] += 1

        del images, labels, outputs, probs, preds, correct_tensor

# 4. In Báo cáo Tối hậu
overall_accuracy = (overall_correct / overall_total) * 100
print("\n" + "🔥"*30)
print(f"🏆 ĐỘ CHÍNH XÁC SUPER TEST: {overall_accuracy:.2f}% ({overall_correct}/{overall_total})")
print("🔥"*30)

print("\n📊 CHI TIẾT SỨC MẠNH TRÊN TỪNG NHÓM (SUBGROUP PERFORMANCE):")
print("-" * 65)
for group in sorted(subgroup_total.keys()):
    correct = subgroup_correct[group]
    total = subgroup_total[group]
    accuracy = (correct / total) * 100
    print(f"Nhóm: {group:<32} | Chính xác: {accuracy:>6.2f}% ({correct}/{total})")
print("-" * 65)

torch.cuda.empty_cache()

Đang nạp tập Super Test vào bộ nhớ...
Đã nạp xong 10000 ảnh vào ống phóng!

🔍 ĐANG ĐÁNH GIÁ MÔ HÌNH TRÊN 5000 ẢNH HOÀN TOÀN MỚI...


Super Testing:   0%|          | 0/157 [00:00<?, ?it/s]


🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
🏆 ĐỘ CHÍNH XÁC SUPER TEST: 96.90% (9690/10000)
🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥

📊 CHI TIẾT SỨC MẠNH TRÊN TỪNG NHÓM (SUBGROUP PERFORMANCE):
-----------------------------------------------------------------
Nhóm: D01_Samsung_GalaxyS3Mini         | Chính xác: 100.00% (5/5)
Nhóm: D02_Apple_iPhone4s               | Chính xác: 100.00% (2/2)
Nhóm: D03_Huawei_P9                    | Chính xác: 100.00% (4/4)
Nhóm: D04_LG_D290                      | Chính xác: 100.00% (2/2)
Nhóm: D05_Apple_iPhone5c               | Chính xác: 100.00% (2/2)
Nhóm: D06_Apple_iPhone6                | Chính xác: 100.00% (4/4)
Nhóm: D07_Lenovo_P70A                  | Chính xác: 100.00% (3/3)
Nhóm: D08_Samsung_GalaxyTab3           | Chính xác: 100.00% (1/1)
Nhóm: D09_Apple_iPhone4                | Chính xác: 100.00% (2/2)
Nhóm: D10_Apple_iPhone4s               | Chính xác: 100.00% (7/7)
Nhóm: D11_Samsung_GalaxyS3             | Chính xác: 100.00% (3/3)
Nhóm: D12_Sony_XperiaZ1Compact        